In [1]:
!pip install mediapipe opencv-python matplotlib

import sys
import os
sys.path.append(os.path.abspath("code"))

import cv2
import mediapipe as mp
import matplotlib.pyplot as plt

from pose_utils import extract_pose_landmarks, get_arrow_tip_position, get_left_elbow_pixel
from draw_utils import draw_annotations
from posture_analysis import (
    analyze_posture,
    analyze_nocking_and_setup,
    analyze_draw_phase,
    analyze_anchor_and_aiming,
    analyze_release,
    analyze_follow_through
)


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
video_dir = "videos"
output_dir = "outputs"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

video_files = [f for f in os.listdir(video_dir) if f.endswith(".mp4")]

In [3]:
MAX_TRAIL = 15

for video_file in video_files:
    input_path = os.path.join(video_dir, video_file)
    output_path = os.path.join(output_dir, "annotated_" + video_file)

    cap = cv2.VideoCapture(input_path)
    out = None
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    if cap.isOpened():
        out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))

    landmarks_prev = None
    arrow_path = []
    elbow_path = []

    with mp.solutions.pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
        frame_count = 0
        while cap.isOpened():
            success, image = cap.read()
            if not success:
                break

            arrow_tip = get_arrow_tip_position(image)
            if arrow_tip:
                arrow_path.append(arrow_tip)
                if len(arrow_path) > MAX_TRAIL:
                    arrow_path.pop(0)

            landmarks = extract_pose_landmarks(image, pose)
            if landmarks:
                elbow_pt = get_left_elbow_pixel(landmarks, image.shape)
                elbow_path.append(elbow_pt)
                if len(elbow_path) > MAX_TRAIL:
                    elbow_path.pop(0)

            annotated_image = draw_annotations(image.copy(), landmarks, arrow_tip, arrow_path, elbow_path)

            posture_feedback = analyze_posture(landmarks)
            nocking_feedback = analyze_nocking_and_setup(landmarks, arrow_tip, image.shape)
            draw_feedback = analyze_draw_phase(landmarks, elbow_path)
            anchor_feedback = analyze_anchor_and_aiming(landmarks)
            release_feedback = analyze_release(landmarks_prev, landmarks) if landmarks_prev else ""
            follow_feedback = analyze_follow_through(landmarks, arrow_tip)

            landmarks_prev = landmarks

            all_feedback = []
            all_feedback += posture_feedback.split(" | ")
            all_feedback += nocking_feedback.split(" | ")
            all_feedback += draw_feedback.split(" | ")
            all_feedback += anchor_feedback.split(" | ")
            all_feedback += release_feedback.split(" | ") if release_feedback else []
            all_feedback += follow_feedback.split(" | ")

            for i, line in enumerate(all_feedback):
                y_offset = 30 + i * 25
                if y_offset < frame_height - 20:
                    cv2.putText(annotated_image, line, (10, y_offset),
                                cv2.FONT_HERSHEY_TRIPLEX, 0.4, (255, 0, 0), 1, cv2.LINE_AA)

            out.write(annotated_image)
            frame_count += 1

        cap.release()
        out.release()